# TNSM – Coleta, Processamento e Plots (Plotly)

Notebook refatorado: pipeline separado em células e **apenas uma célula de plots (Plotly)**.


## 1) Imports e configuração


In [1]:
# Core
import os
import re
import glob
from typing import Dict, Sequence, Optional, Tuple, List

# Data
import numpy as np
import pandas as pd


## 2) Localização dos resultados e colunas coletadas


In [2]:
# Ajuste aqui o padrão do glob conforme o seu layout de pastas
RESULTS_GLOB = '../../results/results_flows*/*'

# Diretórios (cada um contendo CSVs de execuções)
results_flows_directories = glob.glob(RESULTS_GLOB)
print(f"Diretórios encontrados: {len(results_flows_directories)}")
results_flows_directories[:5]


Diretórios encontrados: 1


['../../results/results_flows/vegeta_s_50_p_6_a_0.99_c_3']

In [3]:
metricas_de_coleta = ['tempo',
                        'cpu_utilization',
                        'gpu_utilization',
                        'cache_utilization',
                        'network_cpu_utilization', 
                        'network_gpu_utilization', 
                        'network_cache_utilization', 
                        'mobile_cpu_utilization', 
                        'mobile_gpu_utilization', 
                        'mobile_cache_utilization',
                        'bandwidth_utilization',
                        'bandwidth_utilization_per_flow',
                        'success',
                        'latency',
                        'decision_time_ms',
                        'running_sfcs',
                        'cpu_saved',
                        'shared_vnfs',
                        'shared_vnfs_per_flow',
                        'cpu_per_flow',
                        'gpu_per_flow',
                        'cache_per_flow',
                        "total_energy_consumption",
                        "jain_fairness_latencia_por_sessao",
                        "acceptance_rate",
                        "energy_consumption_per_flow",
                        "queue_time",
                        "jain_cpu",
                        "jain_gpu",
                        "jain_cache",
                        "jain_bw"
                        ]


## 3) Funções auxiliares de métricas


In [4]:
import pandas as pd
import numpy as np

def jain_fairness_index(x):
    """
    Calcula o Jain's Fairness Index para uma pandas Series (x).
    Retorna 1.0 se todos os valores forem 0 (perfeitamente justo).
    Retorna np.nan se a série estiver vazia ou contiver apenas NaNs.
    """
    # Remover valores NaN para o cálculo
    x = x.dropna()
    
    # Contar o número de elementos
    n = len(x)
    
    # Se não houver elementos (grupo vazio ou só NaN),
    # retornar NaN para ser tratado pelo .fillna(0) posteriormente
    if n == 0:
        return np.nan
    
    # Calcular a soma dos quadrados
    sum_sq = (x * x).sum()
    
    # Caso especial: se a soma dos quadrados for 0, 
    # todos os valores são 0. Isso é perfeitamente justo.
    if sum_sq == 0:
        return 1.0
        
    # Calcular a soma dos valores
    sum_val = x.sum()
    
    # Fórmula de Jain
    numerator = sum_val ** 2
    denominator = n * sum_sq
    
    return numerator / denominator


def calcular_jain_fairness_por_sessao(df):
    """
    Calcula o Jain's Fairness Index da latência por sessão e 
    o adiciona ao DataFrame.

    A sessão é extraída da coluna 'sfc_id' (ex: 'sfc_unique_p5_1' -> sessão '1').
    O índice de Jain para um grupo vazio (ou com apenas NaNs) é 'NaN', que é
    convertido para 0. Um grupo com um único valor terá índice 1.0 (justiça perfeita).

    Argumentos:
        df (pd.DataFrame): O DataFrame de entrada. Deve conter as colunas
                             'sfc_id' e 'latency'.

    Retorna:
        pd.DataFrame: Uma cópia do DataFrame original com a nova coluna
                           'jain_fairness_latencia_por_sessao'.
    """
    # Criar uma cópia para evitar modificar o DataFrame original (boa prática)
    df_modificado = df.copy()

    # 1. Extrair o ID da sessão da coluna 'sfc_id'
    df_modificado['session_id'] = df_modificado['sfc_id'].str.split('_').str[-1]

    # 2. Calcular o Jain's Fairness Index da 'latency' para cada 'session_id'
    #    Esta é a linha que foi alterada:
    df_modificado['jain_fairness_latencia_por_sessao'] = df_modificado.groupby('session_id')['latency'].transform(jain_fairness_index)

    # 3. Substituir NaN por 0 (ocorre em sessões vazias ou com apenas NaNs)
    df_modificado['jain_fairness_latencia_por_sessao'] = df_modificado['jain_fairness_latencia_por_sessao'].fillna(0)

    # 4. Remover coluna temporária
    df_modificado = df_modificado.drop(columns=['session_id'])

    return df_modificado


def calcular_consumo_energia_por_fluxo(df):
    """
    Calcula o consumo de energia por fluxo e o adiciona ao DataFrame.

    A nova coluna se chamará 'energy_consumption_per_flow'.
    O cálculo é 'total_energy_consumption' / 'running_sfcs'.

    Casos onde 'running_sfcs' é 0 terão o resultado definido como 0.0.
    """
    df_modificado = df.copy()
    nova_coluna = 'energy_consumption_per_flow'

    df_modificado[nova_coluna] = np.where(
        df_modificado['running_sfcs'] == 0,
        0.0,
        df_modificado['total_energy_consumption'] / df_modificado['running_sfcs']
    )

    return df_modificado


def calcular_banda_por_fluxo(df):
    """
    Calcula a banda utilizada por fluxo e o adiciona ao DataFrame.

    A nova coluna se chamará 'bandwidth_utilization_per_flow'.
    O cálculo é 'bandwidth_utilization' / 'running_sfcs'.

    Casos onde 'running_sfcs' é 0 terão o resultado definido como 0.0.
    """
    df_modificado = df.copy()
    nova_coluna = 'bandwidth_utilization_per_flow'

    df_modificado[nova_coluna] = np.where(
        df_modificado['running_sfcs'] == 0,
        0.0,
        df_modificado['bandwidth_utilization'] / df_modificado['running_sfcs']
    )

    return df_modificado


def calcular_shared_vnfs_por_fluxo(df):
    """
    Calcula a quantidade de VNFs compartilhadas por fluxo e adiciona ao DataFrame.

    A nova coluna se chamará 'shared_vnfs_per_flow'.
    O cálculo é 'shared_vnfs' / 'running_sfcs'.

    Casos onde 'running_sfcs' é 0 terão o resultado definido como 0.0.
    """
    df_modificado = df.copy()
    nova_coluna = 'shared_vnfs_per_flow'

    df_modificado[nova_coluna] = np.where(
        df_modificado['running_sfcs'] == 0,
        0.0,
        df_modificado['shared_vnfs'] / df_modificado['running_sfcs']
    )

    return df_modificado


## 4) Coleta e consolidação de dados (big_data)


In [5]:
SUCCESS_SFC_ID = "sfc_cache_p4_50"  # critério atual de "simulação OK" (ajuste se necessário)

def normalize_algorithm_name(raw_name: str) -> str:
    """Normaliza nomes curtos (ex.: 'ga' -> 'GA'). Se não casar, retorna o nome original."""
    mapping = {
        "g": "Greedy",
        "ga": "GA",
        "msf": "MSF",
        "goku": "OSCIM",
        "vegeta": "Resilient-OSCIM",
        "musfico": "MuSFiCO",
        "greedyb": "GreedyB",
        "kuririnPPO": "Kuririn PPO",
        "darsppo": "DARSPPO",
        "hephaestus": "hephaestus",
    }
    return mapping.get(raw_name, raw_name)

def colect_data_from_alg_directory(results_flows_directories: Sequence[str]) -> pd.DataFrame:
    """Lê todos os CSVs de todos os diretórios e devolve um DataFrame único com médias por tempo."""
    all_runs: List[pd.DataFrame] = []

    for alg_dir in results_flows_directories:
        # Nome do algoritmo = prefixo do nome da pasta (antes do primeiro '_')
        folder_name = os.path.basename(os.path.normpath(alg_dir))
        alg_prefix = folder_name.split('_')[0]
        alg_name = normalize_algorithm_name(alg_prefix)

        try:
            files = [f for f in os.listdir(alg_dir) if f.lower().endswith('.csv')]
        except FileNotFoundError:
            print(f"[AVISO] Diretório não encontrado: {alg_dir}")
            continue

        print(f"Simulação: {alg_name} | CSVs: {len(files)}")

        dir_runs: List[pd.DataFrame] = []
        success_count = 0

        for file in files:
            data_path = os.path.join(alg_dir, file)

            try:
                simulation_df = pd.read_csv(data_path)
            except Exception as e:
                print(f"  [ERRO] Falha ao ler {file}: {e}")
                continue

            # Enriquecimento de métricas
            try:
                simulation_df = calcular_jain_fairness_por_sessao(simulation_df)
                simulation_df = calcular_consumo_energia_por_fluxo(simulation_df)
                simulation_df = calcular_banda_por_fluxo(simulation_df)
                simulation_df = calcular_shared_vnfs_por_fluxo(simulation_df)
            except Exception as e:
                print(f"  [ERRO] Falha ao calcular extras em {file}: {e}")
                continue

            # Tempo relativo
            if "timestamp" not in simulation_df.columns or simulation_df.empty:
                continue
            primeiro_tempo = simulation_df["timestamp"].iloc[0]
            simulation_df["tempo"] = simulation_df["timestamp"] - primeiro_tempo

            # Critério atual de sucesso
            simulation_is_success = ("sfc_id" in simulation_df.columns) and (SUCCESS_SFC_ID in set(simulation_df["sfc_id"].dropna().astype(str).values))

            if not simulation_is_success:
                continue

            success_count += 1

            # Subset de colunas (pode dar KeyError se faltar algo)
            try:
                simulation_df = simulation_df[metricas_de_coleta].copy()
            except KeyError as e:
                missing = sorted(set(metricas_de_coleta) - set(simulation_df.columns))
                print(f"  [ERRO] {file}: faltam colunas ({len(missing)}): {missing[:8]}{'...' if len(missing) > 8 else ''}")
                continue

            simulation_df["tempo"] = simulation_df["tempo"].astype(int)
            simulation_df = simulation_df.replace("None", pd.NA)

            # Prepara latência (garante numérico)
            latency_col = simulation_df[["tempo", "latency"]].dropna()
            if not latency_col.empty:
                latency_col["latency"] = pd.to_numeric(latency_col["latency"], errors="coerce")

            # Média cumulativa de sucesso (se existir coluna 'success')
            if "success" in simulation_df.columns:
                cumulative_sum_success = 0.0
                cumulative_avg_success = []
                for i, value in enumerate(pd.to_numeric(simulation_df["success"], errors="coerce").fillna(0.0).values):
                    cumulative_sum_success += float(value)
                    cumulative_avg_success.append(cumulative_sum_success / (i + 1))
                simulation_df["success"] = cumulative_avg_success

            # Agrega por tempo
            simulation_df = simulation_df.groupby("tempo", as_index=False).mean(numeric_only=True)

            # Reindex para preencher tempos faltantes 0..max
            tempo_max = int(simulation_df["tempo"].max()) if not simulation_df.empty else 0
            df_mean = simulation_df.set_index("tempo").reindex(range(tempo_max + 1)).ffill().reset_index()

            # Sobrescreve latência (se existir)
            if not latency_col.empty:
                latency_df = latency_col.groupby("tempo", as_index=False).mean(numeric_only=True)
                latency_df = latency_df.set_index("tempo").reindex(range(tempo_max + 1)).ffill().reset_index()
                df_mean["latency"] = latency_df["latency"]

            df_mean["algorithm"] = alg_name

            # Trava em 1000s (como no notebook original)
            df_mean = df_mean.iloc[:1000].copy()

            dir_runs.append(df_mean)
            all_runs.append(df_mean)

        if dir_runs:
            dir_df = pd.concat(dir_runs, ignore_index=True)
            print(f"  Simulações OK: {success_count} | Nulos: {int(dir_df.isnull().sum().sum())}")
        else:
            print("  [INFO] Nenhuma simulação OK neste diretório.")

    if not all_runs:
        print("[ERRO] Nenhum dado coletado.")
        return pd.DataFrame()

    big_data = pd.concat(all_runs, ignore_index=True)
    return big_data


In [6]:
# Executa a coleta
big_data = colect_data_from_alg_directory(results_flows_directories)
print("big_data shape:", big_data.shape)
big_data.head()


Simulação: Resilient-OSCIM | CSVs: 23
  [ERRO] Falha ao calcular extras em 202603171606250373636174.csv: 'sfc_id'
  [INFO] Nenhuma simulação OK neste diretório.
[ERRO] Nenhum dado coletado.
big_data shape: (0, 0)


""


## 5) Pré-processamento e features derivadas


In [7]:
# Normalizações (percentuais)
for col in ["acceptance_rate", "cpu_per_flow", "gpu_per_flow"]:
    if col in big_data.columns:
        big_data[col] = big_data[col] / 100.0

# Filtra a partir de 200s
if "tempo" in big_data.columns:
    print(f"Linhas originais: {len(big_data)}")
    big_data = big_data[big_data["tempo"] >= 200].copy()
    print(f"Linhas após tempo>=200: {len(big_data)}")

# CPU saved (mantido do notebook original)
if all(c in big_data.columns for c in ["cpu_saved", "running_sfcs"]):
    big_data["CPU Salva por SFC"] = np.where(big_data["running_sfcs"] == 0, 0.0, big_data["cpu_saved"] / big_data["running_sfcs"])
    big_data["CPU Salva por Servidor"] = big_data["cpu_saved"] / 35.0

# Eficiências (por fluxo)
if all(c in big_data.columns for c in ["cpu_utilization", "running_sfcs"]):
    big_data["eficiencia de cpu"] = np.where(big_data["running_sfcs"] == 0, 0.0, big_data["cpu_utilization"] / big_data["running_sfcs"])
if all(c in big_data.columns for c in ["gpu_utilization", "running_sfcs"]):
    big_data["eficiencia de gpu"] = np.where(big_data["running_sfcs"] == 0, 0.0, big_data["gpu_utilization"] / big_data["running_sfcs"])
if all(c in big_data.columns for c in ["bandwidth_utilization", "running_sfcs"]):
    big_data["eficiencia de banda"] = np.where(big_data["running_sfcs"] == 0, 0.0, big_data["bandwidth_utilization"] / big_data["running_sfcs"])
if all(c in big_data.columns for c in ["cache_utilization", "running_sfcs"]):
    big_data["eficiencia de cache"] = np.where(big_data["running_sfcs"] == 0, 0.0, big_data["cache_utilization"] / big_data["running_sfcs"])

# QoS (mantido do original)
def calcular_pontuacao_latencia(latencia):
    if pd.isna(latencia):
        return 0
    return -1 if latencia > 6 else 2

def calcular_pontuacao_success(success):
    return success * 10

if "latency" in big_data.columns:
    big_data["pontuacao_latencia"] = big_data["latency"].apply(calcular_pontuacao_latencia)
if "success" in big_data.columns:
    big_data["pontuacao_success"] = big_data["success"].apply(calcular_pontuacao_success)

if all(c in big_data.columns for c in ["pontuacao_latencia", "pontuacao_success"]):
    big_data["QoS"] = big_data["pontuacao_latencia"] + big_data["pontuacao_success"]

big_data.head()


""


## 6) Agregação por algoritmo (dados_processados)


In [8]:
# Algoritmos a analisar (IDs devem bater com big_data['algorithm'])
algoritmos = [
    "hephaestusMaskablePPO",
    "darsppoMaskablePPO",
    "kuririnMaskablePPO",
    "GA",
]

dados_processados: Dict[str, pd.DataFrame] = {}

def process_data(data: pd.DataFrame) -> pd.DataFrame:
    return data.groupby("tempo").mean(numeric_only=True)

for alg in algoritmos:
    dados_filtrados = big_data[big_data["algorithm"] == alg].copy()
    if dados_filtrados.empty:
        print(f"[AVISO] Sem dados para algoritmo: {alg}")
        continue

    dados_filtrados = dados_filtrados.drop(columns=["algorithm"], errors="ignore")
    dados_processados[alg] = process_data(dados_filtrados)

list(dados_processados.keys())


KeyError: 'algorithm'

## 7) CÉLULA ÚNICA DE PLOTS (equivalente à célula 19) – Plotly

Tudo que gera figuras fica aqui.


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def create_boxplot(
    data_series_dict: Dict[str, Sequence],
    yaxis_title: str = 'Y Axis',
    xaxis_title: str = 'Time (s)',
    steps: Optional[int] = 200,
    fill_missing: bool = True,
    bfill_leading: bool = True,
    show: bool = True,
    pdf_filename: Optional[str] = None,
    png_filename: Optional[str] = None,
    width: int = 950,
    height: int = 600
) -> Tuple[Optional[go.Figure], pd.DataFrame]:
    """
    Desenha boxplots por janelas para um número arbitrário de séries de dados (Plotly).

    - Cada série é convertida para pd.Series numérica.
    - 'steps' define o tamanho da janela (ex.: 200s).
    - Exportação PDF/PNG usa fig.write_image (requer kaleido). PDF tem fallback para HTML.
    """

    def _as_series(y: Sequence) -> pd.Series:
        if y is None:
            return pd.Series(dtype="float64")
        s = pd.Series(y, dtype="float64")
        return pd.to_numeric(s, errors="coerce")

    def _window_labels(n: int, step: Optional[int]) -> np.ndarray:
        if n == 0 or step is None or step <= 0:
            return np.arange(n, dtype=int)
        groups = np.repeat(np.arange((n + step - 1) // step), step)[:n]
        return (groups + 1) * step

    frames: List[pd.DataFrame] = []
    warnings: List[str] = []

    for label, data in data_series_dict.items():
        s = _as_series(data)
        if s.empty:
            continue

        if fill_missing:
            s = s.ffill()
            if bfill_leading and s.isna().any():
                s = s.bfill()

        if s.dropna().empty:
            warnings.append(f"[AVISO] Série '{label}' ignorada (todos os valores são NaN).")
            continue

        s = s.dropna()
        time_labels = _window_labels(len(s), steps)

        frames.append(pd.DataFrame({
            xaxis_title: time_labels,
            yaxis_title: s.values,
            "Algorithm": label
        }))

    if not frames:
        print("[ERRO] Nenhuma série válida para plotar.")
        return None, pd.DataFrame()

    df_plot = pd.concat(frames, ignore_index=True)

    for msg in warnings:
        print(msg)

    fig = px.box(df_plot, x=xaxis_title, y=yaxis_title, color="Algorithm")
    fig.update_traces(quartilemethod="exclusive")

    unique_x = sorted(df_plot[xaxis_title].unique())
    fig.update_xaxes(categoryorder="array", categoryarray=unique_x)

    fig.update_layout(
        yaxis=dict(title=yaxis_title, showline=True, showgrid=True),
        xaxis=dict(title=xaxis_title, showline=True, showgrid=True),
        legend_title=None,
        margin=dict(l=90, r=20, b=70, t=40),
        width=width, height=height,
        template="plotly_white",
        legend=dict(x=0.5, y=1.12, xanchor='center', yanchor='bottom', orientation='h'),
        font=dict(family="Arial", size=18, color="Black"),
    )

    if show:
        fig.show()

    # Exportação
    if pdf_filename:
        try:
            fig.write_image(pdf_filename)
            print(f"PDF salvo em: {pdf_filename}")
        except Exception as e:
            html_dir = "HTMLs_Fallback"
            os.makedirs(html_dir, exist_ok=True)
            safe_base = re.sub(r"[^\w\-]+", "_", os.path.splitext(pdf_filename)[0])
            html_out = os.path.join(html_dir, f"{safe_base}.html")
            print(f"Falha ao exportar PDF ({e.__class__.__name__}). Salvando HTML: {html_out}")
            fig.write_html(html_out, include_plotlyjs="cdn")

    if png_filename:
        output_dir = "img_salvas"
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.basename(png_filename)
        save_path = os.path.join(output_dir, base_name)
        try:
            fig.write_image(save_path)
            print(f"PNG salvo em: {save_path}")
        except Exception as e:
            print(f"Falha ao exportar PNG ({e.__class__.__name__}): {e}")

    return fig, df_plot


# -----------------------------
# (A) Boxplots por janelas
# -----------------------------
metricas = {
    "SL (ms)": "latency",
    "SAR (%)": "acceptance_rate",
    "DT (ms)": "decision_time_ms",
    "CPUUpF (%)": "cpu_per_flow",
    "GPUUpF (%)": "gpu_per_flow",
    "CUpF. (%)": "cache_per_flow",
    "BUpF (%)": "bandwidth_utilization_per_flow",
    "# of MSSpF": "shared_vnfs_per_flow",
    "PCpF (W)": "energy_consumption_per_flow",
    "JFI": "jain_fairness_latencia_por_sessao",
    "QT (s)": "queue_time",
}

legendas = {
    "kuririnMaskablePPO": "INOMMUS",
    "darsppoMaskablePPO": "DA-RSPPO",
    "hephaestusMaskablePPO": "Hephaestus",
    "GA": "OSCIM",
}

SHOW_PLOTS = True      # True = mostra interativo; False = só prepara/exporta
SAVE_PNG = False       # True = salva PNG em img_salvas/
SAVE_PDF = False       # True = tenta salvar PDF (kaleido); fallback HTML

WINDOW_STEPS = 200

for titulo_grafico, coluna_df in metricas.items():
    multiplicador = 100 if "%" in titulo_grafico else 1

    dados_para_plotar: Dict[str, Sequence] = {}
    for alg_id, legenda_nome in legendas.items():
        if alg_id in dados_processados and coluna_df in dados_processados[alg_id].columns:
            dados_para_plotar[legenda_nome] = dados_processados[alg_id][coluna_df].values * multiplicador
        else:
            print(f"[AVISO] Sem dados para ({alg_id}, {coluna_df}).")

    if not dados_para_plotar:
        continue

    nome_base = re.sub(r"[^\w\-]+", "_", coluna_df)
    pdf_out = f"boxplot_{nome_base}.pdf" if SAVE_PDF else None
    png_out = f"boxplot_{nome_base}.png" if SAVE_PNG else None

    create_boxplot(
        data_series_dict=dados_para_plotar,
        yaxis_title=titulo_grafico,
        xaxis_title=f"Window (step={WINDOW_STEPS}s)",
        steps=WINDOW_STEPS,
        show=SHOW_PLOTS,
        pdf_filename=pdf_out,
        png_filename=png_out,
    )


# -----------------------------
# (B) (Opcional) Boxplot binned (Plotly)
# -----------------------------
def create_binned_boxplot_plotly(
    df_full: pd.DataFrame,
    x_col: str,
    y_col: str,
    algorithm_col: str = "algorithm",
    bin_size: float = 10.0,
    show: bool = True,
    width: int = 1100,
    height: int = 550,
) -> Optional[go.Figure]:
    """Boxplot de y por bins de x, com cor por algoritmo (Plotly)."""
    if df_full.empty or x_col not in df_full.columns or y_col not in df_full.columns or algorithm_col not in df_full.columns:
        print("[AVISO] create_binned_boxplot_plotly: colunas ausentes ou df vazio.")
        return None

    x = pd.to_numeric(df_full[x_col], errors="coerce")
    y = pd.to_numeric(df_full[y_col], errors="coerce")
    alg = df_full[algorithm_col].astype(str)

    dfp = pd.DataFrame({x_col: x, y_col: y, algorithm_col: alg}).dropna()
    if dfp.empty:
        print("[AVISO] create_binned_boxplot_plotly: sem dados após dropna.")
        return None

    # bins
    xmin, xmax = dfp[x_col].min(), dfp[x_col].max()
    if xmin == xmax:
        dfp["bin"] = f"[{xmin:.2f}]"
    else:
        edges = np.arange(np.floor(xmin / bin_size) * bin_size, np.ceil(xmax / bin_size) * bin_size + bin_size, bin_size)
        dfp["bin"] = pd.cut(dfp[x_col], bins=edges, include_lowest=True).astype(str)

    fig = px.box(dfp, x="bin", y=y_col, color=algorithm_col)
    fig.update_traces(quartilemethod="exclusive")
    fig.update_layout(
        template="plotly_white",
        width=width, height=height,
        xaxis_title=f"{x_col} (binned, size={bin_size})",
        yaxis_title=y_col,
        legend_title=None,
        legend=dict(x=0.5, y=1.12, xanchor='center', yanchor='bottom', orientation='h'),
        margin=dict(l=80, r=20, b=90, t=40),
        font=dict(family="Arial", size=18, color="Black"),
    )
    fig.update_xaxes(tickangle=30)

    if show:
        fig.show()
    return fig

# Exemplo de uso (descomente se quiser gerar):
# create_binned_boxplot_plotly(
#     df_full=big_data,
#     x_col="bandwidth_utilization",
#     y_col="acceptance_rate",
#     bin_size=10.0,
#     show=SHOW_PLOTS
# )
